# 🏠 Actividad integradora — Unidades 4 y 5
### Python para Ciencia de Datos | UADE
#### Dataset: Airbnb Buenos Aires (alojamientos + hosts)

---

En esta actividad integramos todo lo visto en las unidades 4 y 5:

| Parte | Tema | Herramientas |
|---|---|---|
| 0 | Carga, limpieza y **left join** | `pd.read_csv`, `pd.merge(how="left")` |
| A | **Estadística descriptiva** sobre `hosts` | media, mediana, moda, desvío, cuartiles, RIC, histograma, boxplot |
| B | **Análisis bivariado** sobre el dataset combinado | scatterplot, boxplot por categoría, pairplot, `corr()`, heatmap |
| C | **Agregaciones** para replicar gráficos de [Inside Airbnb — Buenos Aires](https://insideairbnb.com/es/buenos-aires/) | `groupby`, `agg`, `value_counts`, `pd.cut` |

> 💡 En la U4 ya hicimos la estadística descriptiva de **alojamientos**, así que ahora el foco univariado está puesto en los **hosts**.

En cada sección vas a encontrar celdas **✍️ Para responder**: escribí tu interpretación con tus palabras. Un gráfico sin conclusión no es un análisis.

# 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings generales para visualizar
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

# 2. Cargar datasets

In [ ]:
# Load dataset airbnb_jr.csv con datos de alojamientos
alojamientos = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv")
print(f"Dataset airbnb_jr loaded {alojamientos.shape}")

# Load dataset airbnb_hosts con datos de los hosts
hosts = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_hosts.csv")
print(f"Dataset airbnb_hosts loaded {hosts.shape}")

# 3. Data quality & transformation

## 3.1 Alojamientos

In [ ]:
# normalizar columnas
alojamientos.columns = (
    alojamientos.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)

# drop duplicados
alojamientos.drop_duplicates(inplace=True)

# renombrar columna neighbourhood a barrio
alojamientos.rename(
    columns={
        "neighbourhood": "barrio",
    },
    inplace=True,
)

# drop NaN
alojamientos.dropna(inplace=True)

# transformar price de object a float
try:
    alojamientos["price"] = alojamientos["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    alojamientos["price"] = pd.to_numeric(alojamientos["price"], errors="coerce")
    alojamientos["price_usd"] = round(alojamientos["price"] / 1500, 2)
except Exception as e:
    print(f"No se pudo transformar price {e}")

# transformar last_review a datetime
try:
    alojamientos["last_review"] = pd.to_datetime(alojamientos["last_review"], format="%Y-%m-%d", errors="coerce")
except Exception as e:
    print(f"No se pudo transformar last_review {e}")

print(f"alojamientos limpio: {alojamientos.shape}")

In [ ]:
# alojamientos.info()
# alojamientos.head(3)
# alojamientos.describe()
# alojamientos.describe(include="object")

## 3.2 Hosts

In [ ]:
# cantidad de registros duplicados
print(f"Registros duplicados en hosts: {hosts.duplicated().sum()}")

# visualizar registros duplicados
# hosts[hosts.duplicated(keep=False)].sort_values(by="host_id")

# drop registros duplicados
hosts.drop_duplicates(inplace=True)

# validamos que ahora host_id sea una clave única (1 fila por host)
print(f"host_id repetidos luego de limpiar: {hosts['host_id'].duplicated().sum()}")
print(f"hosts limpio: {hosts.shape}")

In [ ]:
# hosts.info()
# hosts.head(3)
# hosts.describe()
# hosts.describe(include="object")

# 4. Left Join alojamientos - hosts

Usamos **left join** porque la unidad de análisis es el **alojamiento**: queremos conservar todos los alojamientos, tengan o no perfil de host.

> 💡 `validate="m:1"` le pide a pandas que verifique que la relación sea **muchos alojamientos → 1 host**. Si `hosts` tuviera `host_id` repetidos, el merge fallaría en vez de duplicar filas silenciosamente.

In [ ]:
merge = pd.merge(alojamientos, hosts, on="host_id", how="left")

print(f"alojamientos: {alojamientos.shape[0]} filas")
print(f"LEFT JOIN:    {merge.shape[0]} filas  <- debe coincidir con alojamientos")

In [ ]:
# merge.info()
# merge.head(3)
# merge.describe()
# merge.describe(include="object")

In [ ]:
# ¿Cuántos alojamientos tiene cada host? (dentro de nuestra muestra)
merge.groupby("host_id").size().sort_values(ascending=False).head()

---
# Parte A — Estadística descriptiva sobre `hosts`

Trabajamos sobre el dataset **`hosts`** completo (no sobre el merge), porque la unidad de observación ahora es **el anfitrión**.

| Variable | Tipo | Qué vamos a mirar |
|---|---|---|
| `hosts_time_as_user_years` | cuantitativa discreta | posición central, dispersión, histograma, boxplot |
| `host_identity_verified` | cualitativa | frecuencias y moda |
| `host_location` | cualitativa | moda, top ubicaciones |
| `host_about` | texto | ¿completó su descripción? → la convertimos en variable cualitativa |

## A.1 Feature engineering sobre hosts

Antes de describir, creamos algunas variables más fáciles de analizar:

* `verificado`: `host_identity_verified` pasa de `"t"/"f"` a booleano.
* `tiene_about`: ¿el host escribió una descripción? (`host_about` no nulo y distinto de `"-"`).
* `es_local`: ¿el host declara vivir en Buenos Aires? (`host_location` contiene `"Buenos Aires"`).

In [ ]:
# Crear la variable verificado como booleana:
# True si hosts["host_identity_verified"] == "t" sino False
hosts["verificado"] = hosts["host_identity_verified"] == "t"

In [ ]:
# Crear la variable tiene_about como booleana:
# Correr un describe para ver el valor m◙s frecuente
# hosts["host_about"].describe()
hosts["tiene_about"] = hosts["host_about"].notna() & (hosts["host_about"].str.strip() != "-")

In [ ]:
# Crear la variable es_local como booleana:

# hosts["host_location"].value_counts()
hosts["es_local"] = hosts["host_location"].str.contains("buenos aires", case=False, na=False)

In [ ]:
# Mostramos algunas de las variables generadas
hosts[["host_name", "host_location", "verificado", "tiene_about", "es_local", "hosts_time_as_user_years"]].head()

## A.2 Variables cualitativas: frecuencias y moda

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Usá value_counts(normalize=True) sobre verificado, tiene_about y es_local

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: mode()[0] y value_counts().head(10) sobre host_location


✍️ **Para responder:**
1. ¿Qué porcentaje de hosts tiene la identidad verificada? ¿Te sirve esta variable para diferenciar hosts?
2. ¿Qué proporción de hosts declara vivir fuera de Buenos Aires? ¿Qué ubicaciones extranjeras aparecen?

## A.3 Medidas de tendencia central — `hosts_time_as_user_years`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: mean(), median() y mode()[0] sobre hosts_time_as_user_years


## A.4 Medidas de dispersión y cuartiles

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: std(), quantile(0.25/0.75), RIC = Q3 - Q1 y criterio de Tukey


In [ ]:
hosts["hosts_time_as_user_years"].describe()

## A.5 Histograma y boxplot

Reutilizamos la función `plot_histograma_full` de la U4.

In [ ]:
def plot_histograma_full(df, variable, desvio=False, bins="auto"):
    # Completar y borrar pass
    pass


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Histograma con plot_histograma_full y boxplot con sns.boxplot en dos subplots


✍️ **Para responder:**
1. ¿La distribución es simétrica? ¿Por qué la media es **menor** que la mediana?
2. ¿Ves más de un "pico" en el histograma? ¿Qué podría explicar un grupo grande de hosts muy nuevos (0–3 años)?
3. ¿Hay outliers según el criterio de Tukey?

## A.6 ¿Cambia la antigüedad según el perfil del host?

Combinamos una variable cualitativa con una cuantitativa: `groupby` + `describe()` y boxplots por grupo.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("es_local")["hosts_time_as_user_years"].describe()


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Dos boxplots: x=hosts_time_as_user_years, y=es_local / tiene_about


✍️ **Para responder:** ¿Los hosts que completaron su descripción son más antiguos? ¿Y los locales vs. los que viven afuera?

---
# Parte B — Análisis bivariado sobre el dataset combinado (`merge`)

Ahora la unidad de observación vuelve a ser **el alojamiento**, pero enriquecido con los datos de su host.

Preguntas que vamos a responder:
1. ¿Los hosts con más antigüedad tienen mejores `review_scores`?
2. ¿El precio aumenta con la cantidad de `bedrooms`?
3. ¿Los alojamientos con más reseñas tienen mejor puntaje?
4. ¿Qué variables numéricas están más correlacionadas entre sí?

## B.1 Preparar el dataset: outliers de precio

Antes de relacionar variables, revisamos `price_usd`. Un solo precio extremo puede "aplastar" un scatterplot y distorsionar la correlación de Pearson (que es sensible a outliers).

In [ ]:
merge["price_usd"].describe()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Calculá Q1, Q3, RIC y filtrá merge con price_usd <= Q3 + 1.5*RIC en un nuevo df
# LLama al nuevo dataframe df


In [ ]:
# Traemos también las features de hosts creadas en la Parte A
df["verificado"] = hosts["host_identity_verified"] == "t"
df["es_local"] = df["host_location"].str.contains("Buenos Aires", case=False, na=False)
df["tiene_about"] = df["host_about"].notna() & (df["host_about"].str.strip() != "-")

## B.2 `review_scores` vs `hosts_time_as_user_years`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: scatterplot + boxplot con x=hosts_time_as_user_years, y=review_scores; luego .corr()


✍️ **Para responder:** Según la escala vista en la U4 (0–0.25 sin relación, 0.25–0.5 débil…), ¿qué tipo de relación hay? ¿Por qué casi todos los puntos están arriba de 4.5? (pista: mirá `df["review_scores"].describe()`).

## B.3 `price_usd` vs `bedrooms`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: scatterplot + boxplot con x=bedrooms, y=price_usd; luego .corr()


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("bedrooms").agg(cantidad=..., precio_mediano=..., precio_promedio=...)


✍️ **Para responder:** ¿El precio crece con los dormitorios? ¿Qué tan confiables son las conclusiones para 4 o más dormitorios, viendo la columna `cantidad`?

## B.4 `review_scores` vs `number_reviews`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: scatterplot x=number_reviews, y=review_scores, hue="es_local"


✍️ **Para responder:** ¿Qué pasa con la **dispersión** del puntaje a medida que aumentan las reseñas? ¿Por qué un alojamiento con 1 o 2 reseñas puede tener puntajes extremos?

## B.5 Variable cualitativa vs cuantitativa: precio según perfil del host

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: boxplot price_usd por es_local y kdeplot con hue=tiene_about


## B.6 Pairplot

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: sns.pairplot(data=df, vars=[...5 variables numéricas...])


## B.7 Matriz de correlación y mapa de calor

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: df[v_numericas].corr()


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", vmin=-1, vmax=1, center=0)


✍️ **Para responder:**
1. ¿Cuál es el par de variables con la correlación más fuerte? Clasificala según la escala de la U4.
2. ¿Encontraste alguna relación entre variables de **alojamientos** y variables de **hosts**? ¿Qué nos dice eso sobre el valor del merge?
3. Correlación ≠ causalidad: elegí un par correlacionado y proponé una variable que no está en el dataset y que podría explicar ambas.

---
# Parte C — Agregaciones: replicando Inside Airbnb

El sitio [Inside Airbnb — Buenos Aires](https://insideairbnb.com/es/buenos-aires/) publica un tablero con indicadores sobre los alojamientos de la ciudad. Vamos a replicar sus gráficos principales con `groupby` / `agg` sobre **nuestra muestra**.

| Gráfico de Inside Airbnb | Qué agregamos |
|---|---|
| **Tipo de habitación** | % de alojamientos por `room_type` y precio promedio |
| **Actividad** | alojamientos con reseña en los últimos 12 meses |
| **Anuncios por anfitrión** | hosts con 1 alojamiento vs. multi-anfitriones |
| **Principales anfitriones** | ranking de hosts por cantidad de alojamientos |
| **Barrios** (mapa) | cantidad y precio por `barrio` + mapa con latitud/longitud |

> ⚠️ Nuestra muestra tiene ~700 alojamientos (Inside Airbnb trabaja con decenas de miles), así que los números van a diferir. Lo importante es **replicar el método**.

Usamos `merge` (con todos los alojamientos) para conteos, y `df` (sin outliers) para precios.

## C.1 Tipo de habitación

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("room_type").agg(alojamientos=("host_id","count"), ...) y luego calcular porcentaje


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: barplot horizontal de porcentaje por room_type con bar_label


> 💡 Inside Airbnb destaca el % de **alojamientos enteros** porque son los que más compiten con el alquiler tradicional de viviendas.

## C.2 Actividad: ¿cuántos alojamientos están activos?

Inside Airbnb considera **activo** a un alojamiento con al menos una reseña en los últimos 12 meses. Como referencia de "hoy" tomamos la fecha de reseña más reciente del dataset.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Comparar last_review con fecha_ref - pd.DateOffset(months=12)


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("activo").agg(...) con count, mean y median


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("anio_last_review").agg(alojamientos=("host_id","count")) + barplot


## C.3 Anuncios por anfitrión: ¿un solo alojamiento o multi-anfitrión?

Primero **agregamos por host** para saber cuántos alojamientos tiene cada uno, y después **clasificamos** con `pd.cut` (convierte una variable numérica en categorías por rangos).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby(["host_id","host_name"]).agg(alojamientos=("barrio","count")) + pd.cut


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Agrupar por categoria y calcular % de hosts y % de alojamientos


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: melt de "% hosts" y "% alojamientos" + barplot con hue="métrica"


✍️ **Para responder:** ¿Por qué el % de alojamientos de los multi-anfitriones es **mayor** que su % de hosts? ¿Por qué a Inside Airbnb le interesa este indicador?

## C.4 Principales anfitriones

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby(["host_id","host_name"]).agg(...).sort_values("alojamientos").head(10)


✍️ **Para responder:** esta tabla usa `merge` (con outliers). ¿Algún host tiene un precio promedio sospechoso? Buscá sus alojamientos con un filtro y decidí: ¿error de carga o alojamiento de lujo? ¿Qué medida usarías en su lugar?

## C.5 Barrios: cantidad y precio

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: groupby("barrio").agg(alojamientos, precio_mediano_usd, puntaje_promedio, pct_hosts_locales)


In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: Filtrar barrios con >= 10 alojamientos y graficar dos barplots


## C.6 El mapa: un scatterplot con latitud y longitud

El gráfico principal de Inside Airbnb es un **mapa**. ¡Un mapa de puntos no es más que un **scatterplot** donde `x = longitude` e `y = latitude`!

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: scatterplot x="longitude", y="latitude", hue="price_usd"


✍️ **Para responder:** ¿En qué zona de la ciudad se concentran los alojamientos? ¿Y los más caros? Relacionalo con la tabla de barrios de C.5.

---
# 🔑 Conclusiones finales

Escribí un párrafo (5–8 líneas) respondiendo:

1. ¿Cómo es el **perfil típico de un host** de Airbnb en Buenos Aires? (Parte A)
2. ¿Qué variables explican mejor el **precio** y el **puntaje** de un alojamiento? ¿Aportaron algo las variables del host? (Parte B)
3. ¿Qué indicador de Inside Airbnb te pareció más revelador y por qué? (Parte C)

| Tema | Dónde lo usamos |
|---|---|
| `drop_duplicates`, `dropna`, conversión de tipos | Sección 3 |
| `pd.merge(how="left", validate="m:1")` | Sección 4 |
| media, mediana, moda, desvío, cuartiles, Tukey | Parte A |
| histograma, boxplot, kdeplot | Partes A y B |
| scatterplot, pairplot, `corr()`, heatmap | Parte B |
| `groupby`, `agg`, `value_counts`, `pd.cut`, `melt` | Parte C |